# 🚀 Modern Tokenization Techniques for Large Language Models

This notebook provides a comprehensive exploration of tokenization techniques used in modern Large Language Models, from traditional methods to cutting-edge approaches.

## 🎯 What You'll Learn:

### 📝 **Text Tokenization:**
1. **Character-Level Tokenization** - Basic building blocks
2. **Word-Level Tokenization** - Traditional approach
3. **Subword Tokenization** - Modern standard
   - Byte-Pair Encoding (BPE)
   - WordPiece (BERT)
   - SentencePiece (T5, GPT)
   - Unigram Language Model
4. **Advanced Techniques**:
   - Tiktoken (GPT-4)
   - T5 Text-to-Text
   - CANINE (Character-based)
   - ByT5 (Byte-level)

### 🖼️ **Multimodal Tokenization:**
5. **Image Tokenization** - Vision Transformers
6. **Audio Tokenization** - Speech processing
7. **Multimodal Fusion** - Combining text, image, and audio

### 🔬 **Analysis & Comparison:**
8. **Performance Metrics** - Speed, compression, OOV handling
9. **Visualization Tools** - Token distributions, efficiency analysis
10. **Best Practices** - When to use which technique

## 🎓 Learning Objectives:

By the end of this notebook, you will:
- Understand the evolution from character to modern subword tokenization
- Implement state-of-the-art tokenization algorithms from scratch
- Know when to use different tokenization strategies
- Build multimodal tokenization systems
- Analyze and compare tokenization performance
- Apply modern tokenization in real-world LLM applications

## 📚 Theoretical Foundation

### 🔍 **What is Tokenization?**

**Tokenization** is the process of converting raw text (or other data) into discrete units called **tokens** that can be processed by machine learning models.

### 📊 **Evolution of Tokenization**

| Era | Technique | Models | Advantages | Limitations |
|-----|-----------|--------|------------|-------------|
| **Early** | Character-level | RNNs | Simple, no OOV | Long sequences, limited semantics |
| **Traditional** | Word-level | Word2Vec, GloVe | Semantic units | OOV problem, large vocab |
| **Modern** | Subword | BERT, GPT, T5 | Balance of both | Complexity in implementation |
| **Current** | Multimodal | CLIP, DALL-E | Cross-modal understanding | Computational complexity |

### 🎯 **Key Challenges:**

1. **Out-of-Vocabulary (OOV)**: Handling unseen words
2. **Vocabulary Size**: Balance between coverage and efficiency
3. **Semantic Preservation**: Maintaining meaning in tokens
4. **Language Diversity**: Supporting multiple languages
5. **Efficiency**: Speed and memory considerations

### 🧮 **Mathematical Foundation:**

Given a text corpus $C$ with vocabulary $V$, tokenization aims to find the optimal segmentation that:

$$\max_{T} P(C|T) \text{ subject to } |T| \leq k$$

Where $T$ is the tokenization scheme, $P(C|T)$ is the likelihood of the corpus given the tokenization, and $k$ is the maximum vocabulary size.

In [ ]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import re
import time
import json
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("📦 All libraries imported successfully!")
print("🎨 Visualization style configured!")
print("🚀 Ready to explore modern tokenization techniques!")

In [ ]:
class TokenizerAnalyzer:
    """
    A comprehensive class for analyzing and comparing different tokenization techniques.
    
    This class provides methods to:
    - Implement various tokenization algorithms
    - Analyze tokenization performance
    - Visualize token distributions
    - Compare different approaches
    """
    
    def __init__(self):
        self.tokenizers = {}
        self.performance_metrics = {}
        
    def add_tokenizer(self, name: str, tokenizer_func, description: str):
        """Add a tokenizer to the analyzer."""
        self.tokenizers[name] = {
            'function': tokenizer_func,
            'description': description
        }
    
    def analyze_performance(self, text: str, tokenizer_name: str) -> Dict:
        """Analyze performance metrics for a tokenizer."""
        if tokenizer_name not in self.tokenizers:
            raise ValueError(f"Tokenizer '{tokenizer_name}' not found")
        
        tokenizer_func = self.tokenizers[tokenizer_name]['function']
        
        # Measure tokenization time
        start_time = time.time()
        tokens = tokenizer_func(text)
        tokenization_time = time.time() - start_time
        
        # Calculate metrics
        metrics = {
            'num_tokens': len(tokens),
            'unique_tokens': len(set(tokens)),
            'tokenization_time': tokenization_time,
            'compression_ratio': len(text) / len(tokens) if len(tokens) > 0 else 0,
            'avg_token_length': np.mean([len(str(token)) for token in tokens]),
            'tokens_per_second': len(tokens) / tokenization_time if tokenization_time > 0 else float('inf')
        }
        
        self.performance_metrics[tokenizer_name] = metrics
        return metrics
    
    def visualize_token_distribution(self, text: str, tokenizer_names: List[str], title: str = "Token Distribution Analysis"):
        """Visualize token distributions for multiple tokenizers."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(title, fontsize=16, fontweight='bold')
        
        # Token count comparison
        ax1 = axes[0, 0]
        token_counts = []
        names = []
        
        for name in tokenizer_names:
            if name in self.tokenizers:
                tokens = self.tokenizers[name]['function'](text)
                token_counts.append(len(tokens))
                names.append(name)
        
        bars = ax1.bar(names, token_counts, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'][:len(names)])
        ax1.set_title('Number of Tokens')
        ax1.set_ylabel('Token Count')
        ax1.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar, count in zip(bars, token_counts):
            ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(token_counts)*0.01,
                    f'{count}', ha='center', va='bottom')
        
        # Compression ratio comparison
        ax2 = axes[0, 1]
        compression_ratios = [len(text) / count for count in token_counts]
        bars = ax2.bar(names, compression_ratios, color=['orange', 'purple', 'brown', 'pink'][:len(names)])
        ax2.set_title('Compression Ratio')
        ax2.set_ylabel('Characters per Token')
        ax2.tick_params(axis='x', rotation=45)
        
        for bar, ratio in zip(bars, compression_ratios):
            ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(compression_ratios)*0.01,
                    f'{ratio:.2f}', ha='center', va='bottom')
        
        # Token length distribution
        ax3 = axes[1, 0]
        for i, name in enumerate(names[:3]):  # Limit to 3 for clarity
            tokens = self.tokenizers[name]['function'](text)
            token_lengths = [len(str(token)) for token in tokens]
            ax3.hist(token_lengths, bins=20, alpha=0.7, label=name, density=True)
        
        ax3.set_title('Token Length Distribution')
        ax3.set_xlabel('Token Length')
        ax3.set_ylabel('Density')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Performance metrics summary
        ax4 = axes[1, 1]
        ax4.axis('off')
        
        summary_text = "Performance Summary:\n\n"
        for name in names:
            if name in self.performance_metrics:
                metrics = self.performance_metrics[name]
                summary_text += f"{name}:\n"
                summary_text += f"  Tokens: {metrics['num_tokens']}\n"
                summary_text += f"  Unique: {metrics['unique_tokens']}\n"
                summary_text += f"  Compression: {metrics['compression_ratio']:.2f}\n\n"
        
        ax4.text(0.05, 0.95, summary_text, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
        
        plt.tight_layout()
        return fig
    
    def compare_tokenizers(self, text: str, tokenizer_names: List[str]) -> pd.DataFrame:
        """Compare multiple tokenizers and return a summary DataFrame."""
        results = []
        
        for name in tokenizer_names:
            if name in self.tokenizers:
                metrics = self.analyze_performance(text, name)
                results.append({
                    'Tokenizer': name,
                    'Tokens': metrics['num_tokens'],
                    'Unique Tokens': metrics['unique_tokens'],
                    'Compression Ratio': metrics['compression_ratio'],
                    'Avg Token Length': metrics['avg_token_length'],
                    'Speed (tokens/sec)': metrics['tokens_per_second'],
                    'Time (ms)': metrics['tokenization_time'] * 1000
                })
        
        return pd.DataFrame(results).round(3)

# Initialize the analyzer
analyzer = TokenizerAnalyzer()
print("🔬 TokenizerAnalyzer initialized successfully!")
print("📊 Ready to analyze and compare tokenization techniques!")

## 1️⃣ Character-Level Tokenization

### 🔤 **Concept:**
Character-level tokenization treats each character as a separate token. This is the simplest form of tokenization.

### ✅ **Advantages:**
- **No OOV problem** - can handle any text
- **Small vocabulary size** - typically 50-100 characters
- **Language agnostic** - works for any language
- **Simple implementation** - easy to understand and implement

### ❌ **Disadvantages:**
- **Long sequences** - text becomes very long
- **Limited semantics** - characters have little meaning alone
- **Poor efficiency** - many tokens needed for simple words

### 🧮 **Mathematical Foundation:**
For text $T = c_1c_2...c_n$ where $c_i$ are characters:
$$\text{CharTokens}(T) = [c_1, c_2, ..., c_n]$$

In [ ]:
def char_tokenize(text: str) -> List[str]:
    """
    Character-level tokenization.
    
    Args:
        text: Input text to tokenize
    
    Returns:
        List of character tokens
    """
    return list(text)

def enhanced_char_tokenize(text: str, include_special: bool = True) -> List[str]:
    """
    Enhanced character-level tokenization with special token handling.
    
    Args:
        text: Input text to tokenize
        include_special: Whether to include special tokens for spaces, newlines
    
    Returns:
        List of character tokens with optional special tokens
    """
    if not include_special:
        return list(text)
    
    tokens = []
    for char in text:
        if char == ' ':
            tokens.append('<SPACE>')
        elif char == '\n':
            tokens.append('<NEWLINE>')
        elif char == '\t':
            tokens.append('<TAB>')
        else:
            tokens.append(char)
    return tokens

# Add to analyzer
analyzer.add_tokenizer(
    'Character', 
    char_tokenize, 
    'Basic character-level tokenization - each character is a token'
)

analyzer.add_tokenizer(
    'Enhanced Character', 
    enhanced_char_tokenize,
    'Character-level with special tokens for whitespace and control characters'
)

# Demonstration
print("🔤 CHARACTER-LEVEL TOKENIZATION DEMONSTRATION")
print("=" * 60)

sample_text = "Hello, World! How are you?"
print(f"Original text: '{sample_text}'")
print(f"Length: {len(sample_text)} characters")

# Basic character tokenization
char_tokens = char_tokenize(sample_text)
print(f"\n📝 Basic Character Tokens ({len(char_tokens)} tokens):")
print(char_tokens)

# Enhanced character tokenization
enhanced_char_tokens = enhanced_char_tokenize(sample_text)
print(f"\n✨ Enhanced Character Tokens ({len(enhanced_char_tokens)} tokens):")
print(enhanced_char_tokens)

# Vocabulary analysis
unique_chars = set(char_tokens)
print(f"\n📊 Vocabulary Analysis:")
print(f"  Unique characters: {len(unique_chars)}")
print(f"  Vocabulary: {sorted(unique_chars)}")
print(f"  Compression ratio: {len(sample_text) / len(char_tokens):.2f} chars/token")

## 2️⃣ Word-Level Tokenization

### 📝 **Concept:**
Word-level tokenization splits text into words, typically using whitespace and punctuation as delimiters.

### ✅ **Advantages:**
- **Semantic units** - words carry meaning
- **Natural boundaries** - matches human understanding
- **Efficient for common words** - shorter sequences
- **Good for traditional NLP** - works well with word embeddings

### ❌ **Disadvantages:**
- **Large vocabulary** - millions of words in real corpora
- **OOV problem** - cannot handle unseen words
- **Poor morphology** - "run" vs "running" treated separately
- **Language specific** - different rules for different languages

### 🧮 **Mathematical Foundation:**
For text $T$ with word boundaries $B$:
$$\text{WordTokens}(T) = \text{split}(T, B)$$

In [ ]:
import string

def word_tokenize(text: str) -> List[str]:
    """
    Basic word-level tokenization using whitespace splitting.
    
    Args:
        text: Input text to tokenize
    
    Returns:
        List of word tokens
    """
    return text.split()

def advanced_word_tokenize(text: str, handle_punctuation: bool = True, lowercase: bool = False) -> List[str]:
    """
    Advanced word-level tokenization with punctuation and case handling.
    
    Args:
        text: Input text to tokenize
        handle_punctuation: Whether to separate punctuation
        lowercase: Whether to convert to lowercase
    
    Returns:
        List of word tokens
    """
    if lowercase:
        text = text.lower()
    
    if handle_punctuation:
        # Add spaces around punctuation
        for punct in string.punctuation:
            text = text.replace(punct, f' {punct} ')
    
    # Split on whitespace and filter empty strings
    tokens = [token for token in text.split() if token.strip()]
    return tokens

def regex_word_tokenize(text: str) -> List[str]:
    """
    Regex-based word tokenization for better handling of contractions and special cases.
    
    Args:
        text: Input text to tokenize
    
    Returns:
        List of word tokens
    """
    # Pattern to match words, contractions, and punctuation
    pattern = r"\b\w+(?:'\w+)?\b|[.,!?;]"
    tokens = re.findall(pattern, text)
    return tokens

# Add to analyzer
analyzer.add_tokenizer(
    'Word (Basic)', 
    word_tokenize, 
    'Basic word-level tokenization using whitespace splitting'
)

analyzer.add_tokenizer(
    'Word (Advanced)', 
    lambda text: advanced_word_tokenize(text, handle_punctuation=True, lowercase=False),
    'Advanced word tokenization with punctuation handling'
)

analyzer.add_tokenizer(
    'Word (Regex)', 
    regex_word_tokenize,
    'Regex-based word tokenization handling contractions and punctuation'
)

# Demonstration
print("📝 WORD-LEVEL TOKENIZATION DEMONSTRATION")
print("=" * 60)

sample_text = "Hello, World! How are you? I'm doing well. It's a beautiful day!"
print(f"Original text: '{sample_text}'")
print(f"Length: {len(sample_text)} characters")

# Basic word tokenization
word_tokens = word_tokenize(sample_text)
print(f"\n📝 Basic Word Tokens ({len(word_tokens)} tokens):")
print(word_tokens)

# Advanced word tokenization
advanced_word_tokens = advanced_word_tokenize(sample_text, handle_punctuation=True)
print(f"\n✨ Advanced Word Tokens ({len(advanced_word_tokens)} tokens):")
print(advanced_word_tokens)

# Regex word tokenization
regex_word_tokens = regex_word_tokenize(sample_text)
print(f"\n🔍 Regex Word Tokens ({len(regex_word_tokens)} tokens):")
print(regex_word_tokens)

# Vocabulary analysis
print(f"\n📊 Vocabulary Analysis:")
print(f"  Basic unique words: {len(set(word_tokens))}")
print(f"  Advanced unique tokens: {len(set(advanced_word_tokens))}")
print(f"  Regex unique tokens: {len(set(regex_word_tokens))}")

# OOV demonstration
print(f"\n🚫 Out-of-Vocabulary (OOV) Problem:")
known_vocab = set(word_tokens)
new_text = "Supercalifragilisticexpialidocious!"
new_tokens = word_tokenize(new_text)
oov_tokens = [token for token in new_tokens if token not in known_vocab]
print(f"  New text: '{new_text}'")
print(f"  OOV tokens: {oov_tokens}")
print(f"  OOV rate: {len(oov_tokens)/len(new_tokens)*100:.1f}%")

## 3️⃣ Byte-Pair Encoding (BPE) - The GPT Standard

### 🧬 **Concept:**
BPE iteratively merges the most frequent pairs of characters/subwords to create a vocabulary of subword units. Used in GPT, GPT-2, GPT-3, and many others.

### ✅ **Advantages:**
- **Handles OOV** - can represent any word using subwords
- **Efficient vocabulary** - good balance of coverage vs size
- **Data-driven** - learns from actual text distributions
- **Language agnostic** - works across languages

### ❌ **Disadvantages:**
- **Complex implementation** - more sophisticated than word-level
- **Training required** - needs corpus to learn merges
- **Greedy approach** - may not find optimal segmentation

### 🧮 **Mathematical Foundation:**
Given corpus $C$, BPE iteratively finds:
$$\text{merge}^* = \arg\max_{(a,b)} \text{freq}(a,b)$$
Until vocabulary size $|V| = k$

In [ ]:
class ModernBPE:
    """
    Modern implementation of Byte-Pair Encoding (BPE) tokenization.
    
    This implementation includes:
    - Efficient pair counting
    - Vocabulary management
    - Token encoding/decoding
    - Merge rule tracking
    """
    
    def __init__(self, vocab_size: int = 1000):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.merges = []
        self.word_freqs = {}
        
    def _get_word_freqs(self, corpus: List[str]) -> Dict[str, int]:
        """Count word frequencies in corpus."""
        word_freqs = Counter()
        for text in corpus:
            words = text.lower().split()
            for word in words:
                # Add end-of-word symbol
                word_freqs[' '.join(list(word)) + ' </w>'] += 1
        return dict(word_freqs)
    
    def _get_pairs(self, word_freqs: Dict[str, int]) -> Dict[Tuple[str, str], int]:
        """Count all adjacent character pairs."""
        pairs = Counter()
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return dict(pairs)
    
    def _merge_vocab(self, pair: Tuple[str, str], word_freqs: Dict[str, int]) -> Dict[str, int]:
        """Merge the most frequent pair in vocabulary."""
        new_word_freqs = {}
        bigram = ' '.join(pair)
        replacement = ''.join(pair)
        
        for word in word_freqs:
            new_word = word.replace(bigram, replacement)
            new_word_freqs[new_word] = word_freqs[word]
        
        return new_word_freqs
    
    def train(self, corpus: List[str], verbose: bool = False) -> None:
        """Train BPE on a corpus of text."""
        # Initialize word frequencies
        self.word_freqs = self._get_word_freqs(corpus)
        
        # Initialize vocabulary with all characters
        vocab = set()
        for word in self.word_freqs:
            vocab.update(word.split())
        
        self.vocab = {char: i for i, char in enumerate(sorted(vocab))}
        
        if verbose:
            print(f"Initial vocabulary size: {len(self.vocab)}")
        
        # Learn merges
        current_word_freqs = self.word_freqs.copy()
        
        while len(self.vocab) < self.vocab_size:
            pairs = self._get_pairs(current_word_freqs)
            if not pairs:
                break
                
            # Find most frequent pair
            best_pair = max(pairs.items(), key=lambda x: x[1])[0]
            
            # Merge the pair
            current_word_freqs = self._merge_vocab(best_pair, current_word_freqs)
            
            # Add to vocabulary and merges
            new_token = ''.join(best_pair)
            self.vocab[new_token] = len(self.vocab)
            self.merges.append(best_pair)
            
            if verbose and len(self.vocab) % 100 == 0:
                print(f"Vocabulary size: {len(self.vocab)}, Latest merge: {best_pair}")
    
    def encode(self, text: str) -> List[str]:
        """Encode text using trained BPE."""
        words = text.lower().split()
        encoded_words = []
        
        for word in words:
            # Convert to character list with end-of-word marker
            word_tokens = list(word) + ['</w>']
            
            # Apply merges in order
            for merge_pair in self.merges:
                i = 0
                while i < len(word_tokens) - 1:
                    if word_tokens[i] == merge_pair[0] and word_tokens[i + 1] == merge_pair[1]:
                        # Merge the pair
                        word_tokens[i] = merge_pair[0] + merge_pair[1]
                        del word_tokens[i + 1]
                    else:
                        i += 1
            
            encoded_words.extend(word_tokens)
        
        return encoded_words
    
    def get_vocab_info(self) -> Dict:
        """Get information about the learned vocabulary."""
        return {
            'vocab_size': len(self.vocab),
            'num_merges': len(self.merges),
            'sample_tokens': list(self.vocab.keys())[:20],
            'sample_merges': self.merges[:10]
        }

def train_and_use_bpe(corpus: List[str], text: str, vocab_size: int = 500) -> List[str]:
    """Convenience function to train BPE and tokenize text."""
    bpe = ModernBPE(vocab_size=vocab_size)
    bpe.train(corpus, verbose=False)
    return bpe.encode(text)

# Demonstration
print("🧬 BYTE-PAIR ENCODING (BPE) DEMONSTRATION")
print("=" * 60)

# Training corpus
training_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "a quick brown fox is quick and brown",
    "the lazy dog sleeps under the tree",
    "quick quick quick brown brown brown",
    "the fox and the dog are friends",
    "brown animals are beautiful and quick"
]

print(f"Training corpus ({len(training_corpus)} sentences):")
for i, sentence in enumerate(training_corpus):
    print(f"  {i+1}. {sentence}")

# Train BPE
print(f"\n🎓 Training BPE with vocabulary size 200...")
bpe = ModernBPE(vocab_size=200)
bpe.train(training_corpus, verbose=True)

# Show vocabulary info
vocab_info = bpe.get_vocab_info()
print(f"\n📚 Vocabulary Information:")
print(f"  Final vocabulary size: {vocab_info['vocab_size']}")
print(f"  Number of merges learned: {vocab_info['num_merges']}")
print(f"  Sample tokens: {vocab_info['sample_tokens']}")
print(f"  Sample merges: {vocab_info['sample_merges'][:5]}")

# Test tokenization
test_text = "the quickest brown fox"
bpe_tokens = bpe.encode(test_text)
print(f"\n🧪 Tokenization Test:")
print(f"  Input: '{test_text}'")
print(f"  BPE tokens ({len(bpe_tokens)}): {bpe_tokens}")

# Add to analyzer
analyzer.add_tokenizer(
    'BPE', 
    lambda text: train_and_use_bpe(training_corpus, text, vocab_size=500),
    'Byte-Pair Encoding - learns subword units by merging frequent pairs'
)

print(f"\n✅ BPE tokenizer added to analyzer!")

## 4️⃣ WordPiece - The BERT Standard

### 🔍 **Concept:**
WordPiece tokenization, developed by Google, uses a maximum likelihood approach to create subword vocabularies. It's the tokenization method used in BERT and its variants.

### ✅ **Advantages:**
- **Likelihood-based** - optimizes for language modeling
- **Prefix markers** - clear subword boundaries (##)
- **Efficient encoding** - greedy longest-match algorithm
- **Proven performance** - works well in BERT family models

### ❌ **Disadvantages:**
- **Training complexity** - requires likelihood computation
- **Language specific** - prefix notation varies
- **Greedy decoding** - may not find optimal segmentation

### 🧮 **Mathematical Foundation:**
WordPiece maximizes likelihood:
$$\mathcal{L} = \sum_{i=1}^{|D|} \log P(x_i)$$
Where $P(x_i) = \prod_{j=1}^{|x_i|} P(t_j)$ for tokens $t_j$

In [ ]:
class ModernWordPiece:
    """
    Modern implementation of WordPiece tokenization used in BERT.
    
    This implementation includes:
    - Maximum likelihood estimation
    - Greedy longest-match encoding
    - Subword prefix handling (##)
    - OOV handling with <unk>
    """
    
    def __init__(self, vocab_size: int = 1000, unk_token: str = "[UNK]", max_input_chars: int = 200):
        self.vocab_size = vocab_size
        self.unk_token = unk_token
        self.max_input_chars = max_input_chars
        self.vocab = {}
        self.word_counts = {}
    
    def _get_word_counts(self, corpus: List[str]) -> Dict[str, int]:
        """Count word frequencies in corpus."""
        word_counts = Counter()
        for text in corpus:
            words = text.lower().split()
            word_counts.update(words)
        return dict(word_counts)
    
    def _generate_subwords(self, word: str) -> List[str]:
        """Generate all possible subwords for a word."""
        subwords = set()
        
        # Add the whole word
        subwords.add(word)
        
        # Add all prefixes and suffixes
        for i in range(1, len(word)):
            # Prefix
            subwords.add(word[:i])
            # Suffix with ## marker
            subwords.add('##' + word[i:])
            
            # All internal substrings as suffixes
            for j in range(i + 1, len(word) + 1):
                subwords.add('##' + word[i:j])
        
        return list(subwords)
    
    def _calculate_likelihood(self, word: str, segmentation: List[str]) -> float:
        """Calculate likelihood of a word segmentation."""
        likelihood = 0.0
        for token in segmentation:
            if token in self.vocab:
                # Use negative log frequency as a simple likelihood proxy
                likelihood += -np.log(self.vocab[token] + 1)
            else:
                likelihood += -10  # Large penalty for unknown tokens
        return likelihood
    
    def train(self, corpus: List[str], verbose: bool = False) -> None:
        """Train WordPiece on a corpus."""
        # Get word counts
        self.word_counts = self._get_word_counts(corpus)
        
        # Generate all possible subwords
        all_subwords = Counter()
        for word, count in self.word_counts.items():
            subwords = self._generate_subwords(word)
            for subword in subwords:
                all_subwords[subword] += count
        
        # Initialize vocabulary with characters and special tokens
        vocab = {self.unk_token: 0}
        char_vocab = set()
        for word in self.word_counts:
            char_vocab.update(word)
        
        for i, char in enumerate(sorted(char_vocab)):
            vocab[char] = i + 1
        
        # Add most frequent subwords up to vocab_size
        sorted_subwords = all_subwords.most_common(self.vocab_size - len(vocab))
        for subword, count in sorted_subwords:
            if len(vocab) >= self.vocab_size:
                break
            vocab[subword] = count
        
        self.vocab = vocab
        
        if verbose:
            print(f"WordPiece vocabulary size: {len(self.vocab)}")
            print(f"Sample subwords: {list(self.vocab.keys())[:20]}")
    
    def encode(self, text: str) -> List[str]:
        """Encode text using WordPiece tokenization."""
        words = text.lower().split()
        tokens = []
        
        for word in words:
            if len(word) > self.max_input_chars:
                tokens.append(self.unk_token)
                continue
            
            # Greedy longest-match algorithm
            word_tokens = []
            start = 0
            
            while start < len(word):
                end = len(word)
                cur_substr = None
                
                # Find longest matching substring
                while start < end:
                    substr = word[start:end]
                    if start > 0:
                        substr = '##' + substr
                    
                    if substr in self.vocab:
                        cur_substr = substr
                        break
                    
                    end -= 1
                
                if cur_substr is None:
                    word_tokens.append(self.unk_token)
                    break
                
                word_tokens.append(cur_substr)
                start = end
            
            tokens.extend(word_tokens)
        
        return tokens
    
    def decode(self, tokens: List[str]) -> str:
        """Decode WordPiece tokens back to text."""
        text = ""
        for token in tokens:
            if token == self.unk_token:
                text += "[UNK]"
            elif token.startswith('##'):
                text += token[2:]
            else:
                if text:
                    text += " "
                text += token
        return text

def train_and_use_wordpiece(corpus: List[str], text: str, vocab_size: int = 500) -> List[str]:
    """Convenience function to train WordPiece and tokenize text."""
    wp = ModernWordPiece(vocab_size=vocab_size)
    wp.train(corpus, verbose=False)
    return wp.encode(text)

# Demonstration
print("🔍 WORDPIECE TOKENIZATION DEMONSTRATION")
print("=" * 60)

# Training corpus (same as BPE for comparison)
print(f"Using the same training corpus as BPE for comparison...")

# Train WordPiece
print(f"\n🎓 Training WordPiece with vocabulary size 300...")
wp = ModernWordPiece(vocab_size=300)
wp.train(training_corpus, verbose=True)

# Test tokenization
test_sentences = [
    "the quickest brown fox",
    "unbelievable performance",
    "machine learning algorithms"
]

print(f"\n🧪 Tokenization Tests:")
for text in test_sentences:
    wp_tokens = wp.encode(text)
    decoded_text = wp.decode(wp_tokens)
    print(f"  Input: '{text}'")
    print(f"  WordPiece ({len(wp_tokens)}): {wp_tokens}")
    print(f"  Decoded: '{decoded_text}'")
    print()

# Show vocabulary analysis
print(f"📚 Vocabulary Analysis:")
print(f"  Vocabulary size: {len(wp.vocab)}")
sample_subwords = [token for token in wp.vocab.keys() if len(token) > 1][:10]
print(f"  Sample subwords: {sample_subwords}")
prefix_tokens = [token for token in wp.vocab.keys() if token.startswith('##')][:5]
print(f"  Sample ## tokens: {prefix_tokens}")

# Add to analyzer
analyzer.add_tokenizer(
    'WordPiece', 
    lambda text: train_and_use_wordpiece(training_corpus, text, vocab_size=500),
    'WordPiece tokenization used in BERT - likelihood-based subword segmentation'
)

print(f"\n✅ WordPiece tokenizer added to analyzer!")

## 5️⃣ SentencePiece - The T5 Standard

### 🌐 **Concept:**
SentencePiece treats text as a sequence of characters and learns subword units without pre-tokenization. It's language-agnostic and used in T5, mT5, and many multilingual models.

### ✅ **Advantages:**
- **Language agnostic** - no pre-tokenization needed
- **Raw text processing** - handles spaces as normal characters
- **Multiple algorithms** - supports BPE and Unigram LM
- **Reversible** - perfect reconstruction of original text
- **Unicode friendly** - handles all character encodings

### ❌ **Disadvantages:**
- **Implementation complexity** - more sophisticated than basic methods
- **Memory intensive** - stores all possible segmentations
- **Training time** - slower than simpler methods

### 🧮 **Mathematical Foundation:**
For Unigram LM variant:
$$P(x) = \prod_{i=1}^{n} P(x_i)$$
Where segmentation is chosen to maximize $P(x)$

In [ ]:
class ModernSentencePiece:
    """
    Modern implementation of SentencePiece tokenization.
    
    This implementation includes:
    - Unigram Language Model approach
    - Character-level fallback
    - Space preservation (▁ symbol)
    - Language-agnostic processing
    """
    
    def __init__(self, vocab_size: int = 1000, character_coverage: float = 0.9995):
        self.vocab_size = vocab_size
        self.character_coverage = character_coverage
        self.vocab = {}
        self.pieces = []
        self.scores = {}
        
    def _normalize_text(self, text: str) -> str:
        """Normalize text by replacing spaces with special symbol."""
        # Replace spaces with ▁ (U+2581)
        return '▁' + text.replace(' ', '▁')
    
    def _get_character_counts(self, corpus: List[str]) -> Counter:
        """Get character frequency counts from corpus."""
        char_counts = Counter()
        for text in corpus:
            normalized = self._normalize_text(text.lower())
            char_counts.update(normalized)
        return char_counts
    
    def _generate_candidate_pieces(self, corpus: List[str], max_piece_length: int = 16) -> List[str]:
        """Generate candidate pieces from the corpus."""
        pieces = set()
        
        for text in corpus:
            normalized = self._normalize_text(text.lower())
            
            # Add all substrings up to max_piece_length
            for i in range(len(normalized)):
                for j in range(i + 1, min(i + max_piece_length + 1, len(normalized) + 1)):
                    piece = normalized[i:j]
                    pieces.add(piece)
        
        return list(pieces)
    
    def _calculate_piece_scores(self, pieces: List[str], corpus: List[str]) -> Dict[str, float]:
        """Calculate scores for pieces using frequency-based approach."""
        piece_counts = Counter()
        total_chars = 0
        
        for text in corpus:
            normalized = self._normalize_text(text.lower())
            total_chars += len(normalized)
            
            for piece in pieces:
                piece_counts[piece] += normalized.count(piece)
        
        # Calculate scores (log probability)
        scores = {}
        for piece, count in piece_counts.items():
            if count > 0:
                scores[piece] = np.log(count / total_chars)
            else:
                scores[piece] = -float('inf')
        
        return scores
    
    def _find_best_segmentation(self, text: str) -> List[str]:
        """Find best segmentation using dynamic programming."""
        n = len(text)
        # dp[i] = (best_score, best_segmentation) for text[:i]
        dp = [(-float('inf'), [])] * (n + 1)
        dp[0] = (0.0, [])
        
        for i in range(1, n + 1):
            for j in range(i):
                piece = text[j:i]
                if piece in self.scores:
                    score = dp[j][0] + self.scores[piece]
                    if score > dp[i][0]:
                        dp[i] = (score, dp[j][1] + [piece])
        
        return dp[n][1] if dp[n][1] else list(text)  # Fallback to characters
    
    def train(self, corpus: List[str], verbose: bool = False) -> None:
        """Train SentencePiece model on corpus."""
        # Get character counts and coverage
        char_counts = self._get_character_counts(corpus)
        total_chars = sum(char_counts.values())
        
        # Select characters based on coverage
        sorted_chars = char_counts.most_common()
        covered_chars = 0
        selected_chars = []
        
        for char, count in sorted_chars:
            selected_chars.append(char)
            covered_chars += count
            if covered_chars / total_chars >= self.character_coverage:
                break
        
        if verbose:
            print(f"Selected {len(selected_chars)} characters covering {covered_chars/total_chars:.4f} of corpus")
        
        # Generate candidate pieces
        candidate_pieces = self._generate_candidate_pieces(corpus)
        if verbose:
            print(f"Generated {len(candidate_pieces)} candidate pieces")
        
        # Calculate piece scores
        all_pieces = selected_chars + candidate_pieces
        self.scores = self._calculate_piece_scores(all_pieces, corpus)
        
        # Select top pieces for vocabulary
        sorted_pieces = sorted(self.scores.items(), key=lambda x: x[1], reverse=True)
        self.pieces = [piece for piece, score in sorted_pieces[:self.vocab_size]]
        self.vocab = {piece: i for i, piece in enumerate(self.pieces)}
        
        if verbose:
            print(f"Final vocabulary size: {len(self.vocab)}")
            print(f"Sample pieces: {self.pieces[:20]}")
    
    def encode(self, text: str) -> List[str]:
        """Encode text using SentencePiece."""
        normalized = self._normalize_text(text.lower())
        return self._find_best_segmentation(normalized)
    
    def decode(self, tokens: List[str]) -> str:
        """Decode SentencePiece tokens back to text."""
        text = ''.join(tokens)
        # Replace ▁ with spaces
        text = text.replace('▁', ' ')
        # Remove leading space
        return text.strip()

def train_and_use_sentencepiece(corpus: List[str], text: str, vocab_size: int = 500) -> List[str]:
    """Convenience function to train SentencePiece and tokenize text."""
    sp = ModernSentencePiece(vocab_size=vocab_size)
    sp.train(corpus, verbose=False)
    return sp.encode(text)

# Demonstration
print("🌐 SENTENCEPIECE TOKENIZATION DEMONSTRATION")
print("=" * 60)

# Training corpus with more diverse examples
multilingual_corpus = training_corpus + [
    "hello world how are you today",
    "natural language processing is fascinating",
    "machine learning models need good tokenization",
    "subword units help with out of vocabulary words"
]

print(f"Training corpus ({len(multilingual_corpus)} sentences):")
for i, sentence in enumerate(multilingual_corpus[-4:]):
    print(f"  {i+1}. {sentence}")
print("  ... (and previous sentences)")

# Train SentencePiece
print(f"\n🎓 Training SentencePiece with vocabulary size 400...")
sp = ModernSentencePiece(vocab_size=400)
sp.train(multilingual_corpus, verbose=True)

# Test tokenization
test_sentences = [
    "the quickest brown fox",
    "unbelievable performance",
    "natural language processing"
]

print(f"\n🧪 Tokenization Tests:")
for text in test_sentences:
    sp_tokens = sp.encode(text)
    decoded_text = sp.decode(sp_tokens)
    print(f"  Input: '{text}'")
    print(f"  SentencePiece ({len(sp_tokens)}): {sp_tokens}")
    print(f"  Decoded: '{decoded_text}'")
    print()

# Show vocabulary analysis
print(f"📚 Vocabulary Analysis:")
print(f"  Vocabulary size: {len(sp.vocab)}")
space_tokens = [token for token in sp.pieces if '▁' in token][:5]
print(f"  Sample space tokens: {space_tokens}")
word_pieces = [token for token in sp.pieces if len(token) > 2 and '▁' not in token][:5]
print(f"  Sample word pieces: {word_pieces}")

# Add to analyzer
analyzer.add_tokenizer(
    'SentencePiece', 
    lambda text: train_and_use_sentencepiece(multilingual_corpus, text, vocab_size=500),
    'SentencePiece tokenization - language-agnostic subword segmentation'
)

print(f"\n✅ SentencePiece tokenizer added to analyzer!")

## 6️⃣ Advanced Modern Techniques

### 🚀 **Tiktoken (GPT-4)**
OpenAI's tiktoken uses a refined BPE approach with optimizations for modern transformer models.

### 🔤 **CANINE (Character-based)**
Google's CANINE operates directly on Unicode characters, eliminating tokenization altogether.

### 📱 **ByT5 (Byte-level)**
Google's ByT5 works at the byte level, making it truly language-agnostic.

In [ ]:
class AdvancedTokenizers:
    """Collection of advanced tokenization techniques."""
    
    @staticmethod
    def tiktoken_style_bpe(text: str, vocab_size: int = 1000) -> List[str]:
        """
        Simplified tiktoken-style BPE with regex pre-tokenization.
        
        This mimics the pattern used in GPT-4's tiktoken tokenizer.
        """
        # Regex pattern similar to tiktoken
        pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        
        # Simplified version - split on word boundaries and punctuation
        import re
        tokens = re.findall(r"\w+|[.,!?;]|\s+", text)
        
        # Apply BPE-style merging (simplified)
        char_tokens = []
        for token in tokens:
            if token.strip():  # Skip whitespace-only tokens
                char_tokens.extend(list(token))
        
        return char_tokens
    
    @staticmethod
    def canine_style_unicode(text: str) -> List[int]:
        """
        CANINE-style tokenization using Unicode codepoints.
        
        Returns Unicode codepoints instead of string tokens.
        """
        return [ord(char) for char in text]
    
    @staticmethod
    def byt5_style_bytes(text: str) -> List[int]:
        """
        ByT5-style byte-level tokenization.
        
        Converts text to UTF-8 bytes.
        """
        return list(text.encode('utf-8'))
    
    @staticmethod
    def morphological_tokenize(text: str) -> List[str]:
        """
        Simple morphological tokenization.
        
        Attempts to split words into morphemes.
        """
        words = text.split()
        morphemes = []
        
        for word in words:
            # Simple prefix/suffix splitting
            if word.endswith('ing'):
                morphemes.extend([word[:-3], 'ing'])
            elif word.endswith('ed'):
                morphemes.extend([word[:-2], 'ed'])
            elif word.endswith('ly'):
                morphemes.extend([word[:-2], 'ly'])
            elif word.startswith('un'):
                morphemes.extend(['un', word[2:]])
            elif word.startswith('re'):
                morphemes.extend(['re', word[2:]])
            else:
                morphemes.append(word)
        
        return morphemes

# Add advanced tokenizers to analyzer
analyzer.add_tokenizer(
    'Tiktoken-style',
    AdvancedTokenizers.tiktoken_style_bpe,
    'GPT-4 style tokenization with regex pre-processing'
)

analyzer.add_tokenizer(
    'Unicode Codepoints',
    lambda text: [str(cp) for cp in AdvancedTokenizers.canine_style_unicode(text)],
    'CANINE-style Unicode codepoint tokenization'
)

analyzer.add_tokenizer(
    'Byte-level',
    lambda text: [str(b) for b in AdvancedTokenizers.byt5_style_bytes(text)],
    'ByT5-style byte-level tokenization'
)

analyzer.add_tokenizer(
    'Morphological',
    AdvancedTokenizers.morphological_tokenize,
    'Simple morpheme-based tokenization'
)

# Demonstration
print("🚀 ADVANCED TOKENIZATION TECHNIQUES DEMONSTRATION")
print("=" * 60)

test_text = "The running dogs quickly jumped over the fence, unbelievably!"
print(f"Test text: '{test_text}'")
print(f"Length: {len(test_text)} characters")

print(f"\n🔍 Advanced Tokenization Results:")

# Tiktoken-style
tiktoken_tokens = AdvancedTokenizers.tiktoken_style_bpe(test_text)
print(f"\n📱 Tiktoken-style ({len(tiktoken_tokens)} tokens):")
print(f"  {tiktoken_tokens}")

# Unicode codepoints
unicode_tokens = AdvancedTokenizers.canine_style_unicode(test_text)
print(f"\n🔤 Unicode Codepoints ({len(unicode_tokens)} tokens):")
print(f"  {unicode_tokens[:20]}... (showing first 20)")

# Byte-level
byte_tokens = AdvancedTokenizers.byt5_style_bytes(test_text)
print(f"\n📱 Byte-level ({len(byte_tokens)} tokens):")
print(f"  {byte_tokens[:20]}... (showing first 20)")

# Morphological
morpho_tokens = AdvancedTokenizers.morphological_tokenize(test_text.lower())
print(f"\n🧬 Morphological ({len(morpho_tokens)} tokens):")
print(f"  {morpho_tokens}")

print(f"\n✅ Advanced tokenizers added to analyzer!")

## 7️⃣ Comprehensive Tokenizer Comparison

Now let's compare all the tokenization techniques we've implemented using our analyzer framework.

In [ ]:
# 🔬 COMPREHENSIVE TOKENIZER COMPARISON
print("🔬 COMPREHENSIVE TOKENIZER COMPARISON")
print("=" * 60)

# Test text with diverse characteristics
comparison_text = """
The quick brown fox jumps over the lazy dog. This sentence contains various 
linguistic challenges: contractions (don't, won't), hyphenated-words, 
numbers (123, 4.56), punctuation!?; and some uncommon words like 
supercalifragilisticexpialidocious. It also includes proper nouns like 
OpenAI and abbreviations like NLP, AI, and ML.
""".strip()

print(f"Test text ({len(comparison_text)} characters):")
print(f"'{comparison_text[:100]}...'")

# Compare all tokenizers
tokenizer_names = [
    'Character', 'Word (Basic)', 'Word (Advanced)', 'BPE', 
    'WordPiece', 'SentencePiece', 'Morphological'
]

print(f"\n📊 Performance Comparison:")
comparison_df = analyzer.compare_tokenizers(comparison_text, tokenizer_names)
print(comparison_df)

# Create visualization
print(f"\n📈 Creating visualization...")
fig = analyzer.visualize_token_distribution(
    comparison_text, 
    tokenizer_names,
    "Comprehensive Tokenization Comparison"
)
plt.show()

# Analysis of results
print(f"\n🎯 Key Insights:")
print(f"  • Character tokenization: Longest sequence ({comparison_df.loc[comparison_df['Tokenizer'] == 'Character', 'Tokens'].values[0]} tokens)")
print(f"  • Word tokenization: Simplest but OOV problems")
print(f"  • Subword methods (BPE, WordPiece, SP): Best balance")
print(f"  • Morphological: Good for specific languages")

# Efficiency analysis
best_compression = comparison_df.loc[comparison_df['Compression Ratio'].idxmax()]
fastest = comparison_df.loc[comparison_df['Speed (tokens/sec)'].idxmax()]

print(f"\n🏆 Winners:")
print(f"  • Best compression: {best_compression['Tokenizer']} ({best_compression['Compression Ratio']:.2f} chars/token)")
print(f"  • Fastest: {fastest['Tokenizer']} ({fastest['Speed (tokens/sec)']:.0f} tokens/sec)")

## 8️⃣ Multimodal Tokenization

### 🖼️ **Vision Tokenization**
Modern vision transformers tokenize images by splitting them into patches.

### 🔊 **Audio Tokenization**
Speech models use various tokenization strategies from raw waveforms to spectrograms.

### 🔄 **Cross-modal Fusion**
Combining different modalities requires careful token alignment and fusion strategies.

In [ ]:
class MultimodalTokenizer:
    """
    Multimodal tokenization for text, images, and audio.
    
    This class demonstrates how different modalities can be tokenized
    and combined for multimodal language models.
    """
    
    def __init__(self):
        self.special_tokens = {
            'text_start': '<TEXT>',
            'text_end': '</TEXT>',
            'image_start': '<IMAGE>',
            'image_end': '</IMAGE>',
            'audio_start': '<AUDIO>',
            'audio_end': '</AUDIO>',
            'patch': '<PATCH_',
            'audio_frame': '<FRAME_'
        }
    
    def tokenize_image_patches(self, image_shape: Tuple[int, int, int], patch_size: int = 16) -> List[str]:
        """
        Tokenize image into patches (Vision Transformer style).
        
        Args:
            image_shape: (height, width, channels)
            patch_size: Size of each patch
        
        Returns:
            List of patch tokens
        """
        height, width, channels = image_shape
        
        # Calculate number of patches
        num_patches_h = height // patch_size
        num_patches_w = width // patch_size
        
        tokens = [self.special_tokens['image_start']]
        
        # Create patch tokens
        for i in range(num_patches_h):
            for j in range(num_patches_w):
                patch_id = i * num_patches_w + j
                tokens.append(f"{self.special_tokens['patch']}{patch_id}>")
        
        tokens.append(self.special_tokens['image_end'])
        return tokens
    
    def tokenize_audio_frames(self, audio_length: float, frame_rate: int = 50) -> List[str]:
        """
        Tokenize audio into temporal frames.
        
        Args:
            audio_length: Length of audio in seconds
            frame_rate: Frames per second
        
        Returns:
            List of audio frame tokens
        """
        num_frames = int(audio_length * frame_rate)
        
        tokens = [self.special_tokens['audio_start']]
        
        for frame_id in range(num_frames):
            tokens.append(f"{self.special_tokens['audio_frame']}{frame_id}>")
        
        tokens.append(self.special_tokens['audio_end'])
        return tokens
    
    def combine_multimodal_tokens(self, text_tokens: List[str], 
                                 image_tokens: List[str] = None,
                                 audio_tokens: List[str] = None) -> List[str]:
        """
        Combine tokens from different modalities.
        
        Args:
            text_tokens: Text tokens
            image_tokens: Image patch tokens
            audio_tokens: Audio frame tokens
        
        Returns:
            Combined multimodal token sequence
        """
        combined = []
        
        # Add text tokens
        combined.append(self.special_tokens['text_start'])
        combined.extend(text_tokens)
        combined.append(self.special_tokens['text_end'])
        
        # Add image tokens if provided
        if image_tokens:
            combined.extend(image_tokens)
        
        # Add audio tokens if provided
        if audio_tokens:
            combined.extend(audio_tokens)
        
        return combined
    
    def create_attention_mask(self, tokens: List[str]) -> Dict[str, List[int]]:
        """
        Create attention masks for different modalities.
        
        Args:
            tokens: Combined multimodal tokens
        
        Returns:
            Dictionary of attention masks for each modality
        """
        text_mask = []
        image_mask = []
        audio_mask = []
        
        current_modality = 'none'
        
        for token in tokens:
            if token == self.special_tokens['text_start']:
                current_modality = 'text'
            elif token == self.special_tokens['image_start']:
                current_modality = 'image'
            elif token == self.special_tokens['audio_start']:
                current_modality = 'audio'
            elif token in [self.special_tokens['text_end'], 
                          self.special_tokens['image_end'],
                          self.special_tokens['audio_end']]:
                current_modality = 'none'
            
            # Set masks
            text_mask.append(1 if current_modality == 'text' else 0)
            image_mask.append(1 if current_modality == 'image' else 0)
            audio_mask.append(1 if current_modality == 'audio' else 0)
        
        return {
            'text_mask': text_mask,
            'image_mask': image_mask,
            'audio_mask': audio_mask
        }

# Demonstration
print("🎭 MULTIMODAL TOKENIZATION DEMONSTRATION")
print("=" * 60)

multimodal = MultimodalTokenizer()

# Example: Image captioning scenario
text = "A brown dog is running in the park"
text_tokens = text.split()  # Simple tokenization for demo
image_shape = (224, 224, 3)  # Standard ImageNet size
patch_size = 16

print(f"🖼️ Image Tokenization:")
print(f"  Image shape: {image_shape}")
print(f"  Patch size: {patch_size}x{patch_size}")

image_tokens = multimodal.tokenize_image_patches(image_shape, patch_size)
print(f"  Generated {len(image_tokens)} image tokens")
print(f"  Sample: {image_tokens[:8]}...")

print(f"\n🔊 Audio Tokenization:")
audio_length = 3.5  # 3.5 seconds
audio_tokens = multimodal.tokenize_audio_frames(audio_length)
print(f"  Audio length: {audio_length} seconds")
print(f"  Generated {len(audio_tokens)} audio tokens")
print(f"  Sample: {audio_tokens[:8]}...")

print(f"\n🔄 Multimodal Fusion:")
combined_tokens = multimodal.combine_multimodal_tokens(
    text_tokens, image_tokens, audio_tokens
)
print(f"  Total combined tokens: {len(combined_tokens)}")
print(f"  Text: {len(text_tokens)}, Image: {len(image_tokens)}, Audio: {len(audio_tokens)}")

# Show token structure
print(f"\n📋 Token Sequence Structure:")
modality_counts = {'TEXT': 0, 'IMAGE': 0, 'AUDIO': 0, 'SPECIAL': 0}
for token in combined_tokens:
    if '<TEXT>' in token or '</TEXT>' in token:
        modality_counts['SPECIAL'] += 1
    elif '<IMAGE>' in token or '</IMAGE>' in token or '<PATCH_' in token:
        if '<IMAGE>' in token or '</IMAGE>' in token:
            modality_counts['SPECIAL'] += 1
        else:
            modality_counts['IMAGE'] += 1
    elif '<AUDIO>' in token or '</AUDIO>' in token or '<FRAME_' in token:
        if '<AUDIO>' in token or '</AUDIO>' in token:
            modality_counts['SPECIAL'] += 1
        else:
            modality_counts['AUDIO'] += 1
    else:
        modality_counts['TEXT'] += 1

for modality, count in modality_counts.items():
    print(f"  {modality}: {count} tokens")

# Create attention masks
attention_masks = multimodal.create_attention_mask(combined_tokens)
print(f"\n🎯 Attention Masks:")
for mask_type, mask in attention_masks.items():
    active_positions = sum(mask)
    print(f"  {mask_type}: {active_positions} active positions out of {len(mask)}")

print(f"\n✅ Multimodal tokenization complete!")
print(f"\n💡 Use Cases:")
print(f"  • Image captioning: text + image")
print(f"  • Visual question answering: text + image")
print(f"  • Speech-to-image: audio + image generation")
print(f"  • Multimodal chatbots: all modalities combined")

## 9️⃣ Best Practices and Guidelines

### 📝 **When to Use Which Tokenizer:**

| Use Case | Recommended Tokenizer | Why |
|----------|----------------------|-----|
| **General LLMs** | BPE or SentencePiece | Best balance of efficiency and coverage |
| **BERT-style models** | WordPiece | Optimized for masked language modeling |
| **Multilingual models** | SentencePiece | Language-agnostic, no pre-tokenization |
| **Character-level tasks** | Character | When fine-grained control needed |
| **Code generation** | BPE with code-specific vocab | Handles code syntax well |
| **Scientific text** | Custom subword + domain vocab | Preserves technical terms |

### ⚡ **Performance Optimization:**

1. **Vocabulary Size**: 8K-32K for most applications
2. **Training Data**: Use representative, diverse corpora
3. **Special Tokens**: Reserve space for task-specific tokens
4. **Preprocessing**: Normalize Unicode, handle whitespace consistently
5. **Evaluation**: Test on downstream tasks, not just perplexity

### 🛡️ **Common Pitfalls:**

- **Inconsistent preprocessing** between training and inference
- **Vocabulary mismatch** between tokenizer and model
- **Language bias** in multilingual settings
- **Overfitting** tokenizer to training data
- **Ignoring edge cases** like empty strings, very long texts

In [ ]:
# 🎓 FINAL COMPREHENSIVE ANALYSIS AND RECOMMENDATIONS
print("🎓 FINAL COMPREHENSIVE ANALYSIS AND RECOMMENDATIONS")
print("=" * 60)

# Create summary analysis
def create_tokenizer_summary():
    """Create a comprehensive summary of all tokenizers."""
    
    summary_data = {
        'Character': {
            'complexity': 1, 'vocab_size': 100, 'oov_handling': 5,
            'efficiency': 2, 'semantic_preservation': 1,
            'use_cases': ['Character-level tasks', 'Morphologically rich languages'],
            'pros': ['No OOV', 'Simple', 'Language agnostic'],
            'cons': ['Long sequences', 'Limited semantics']
        },
        'Word': {
            'complexity': 2, 'vocab_size': 50000, 'oov_handling': 1,
            'efficiency': 4, 'semantic_preservation': 5,
            'use_cases': ['Traditional NLP', 'Small domain-specific tasks'],
            'pros': ['Semantic units', 'Natural boundaries'],
            'cons': ['Large vocabulary', 'OOV problem']
        },
        'BPE': {
            'complexity': 4, 'vocab_size': 32000, 'oov_handling': 5,
            'efficiency': 5, 'semantic_preservation': 4,
            'use_cases': ['GPT family', 'General LLMs', 'Code generation'],
            'pros': ['Good balance', 'Handles OOV', 'Data-driven'],
            'cons': ['Training required', 'Greedy approach']
        },
        'WordPiece': {
            'complexity': 4, 'vocab_size': 30000, 'oov_handling': 5,
            'efficiency': 4, 'semantic_preservation': 4,
            'use_cases': ['BERT family', 'Masked LM tasks'],
            'pros': ['Likelihood-based', 'Clear boundaries', 'Proven'],
            'cons': ['Complex training', 'Language specific']
        },
        'SentencePiece': {
            'complexity': 5, 'vocab_size': 32000, 'oov_handling': 5,
            'efficiency': 5, 'semantic_preservation': 4,
            'use_cases': ['Multilingual models', 'T5 family', 'Any language'],
            'pros': ['Language agnostic', 'No pre-tokenization', 'Reversible'],
            'cons': ['Complex implementation', 'Memory intensive']
        }
    }
    
    return summary_data

summary = create_tokenizer_summary()

# Create comparison matrix
print("📊 TOKENIZER COMPARISON MATRIX")
print("=" * 40)

metrics = ['Complexity', 'Vocab Size', 'OOV Handling', 'Efficiency', 'Semantic Preservation']
print(f"{'Tokenizer':<15} {' '.join(f'{metric:<12}' for metric in metrics)}")
print("-" * 80)

for name, data in summary.items():
    scores = [data['complexity'], data['vocab_size']//1000, data['oov_handling'], 
              data['efficiency'], data['semantic_preservation']]
    score_str = ' '.join(f'{score:<12}' for score in scores)
    print(f"{name:<15} {score_str}")

print("\n📈 RECOMMENDATIONS BY USE CASE")
print("=" * 40)

recommendations = {
    '🤖 General Purpose LLMs': 'BPE or SentencePiece - best balance of all factors',
    '🔍 Search & Understanding': 'WordPiece - optimized for BERT-style models',
    '🌍 Multilingual Applications': 'SentencePiece - language agnostic design',
    '💻 Code Generation': 'BPE with code-specific preprocessing',
    '📚 Domain-Specific Tasks': 'Custom vocabulary with subword method',
    '⚡ Real-time Applications': 'Word-level for speed, BPE for quality',
    '🎯 Few-shot Learning': 'Character or byte-level for maximum flexibility',
    '🔬 Research & Experimentation': 'SentencePiece for reproducibility'
}

for use_case, recommendation in recommendations.items():
    print(f"\n{use_case}:")
    print(f"  {recommendation}")

print("\n🛠️ IMPLEMENTATION CHECKLIST")
print("=" * 40)

checklist = [
    "✅ Choose tokenizer based on use case and requirements",
    "✅ Prepare representative training corpus",
    "✅ Set appropriate vocabulary size (8K-32K typical)",
    "✅ Include special tokens for your task",
    "✅ Implement consistent preprocessing pipeline",
    "✅ Test on diverse inputs including edge cases",
    "✅ Validate on downstream task performance",
    "✅ Document tokenization choices for reproducibility",
    "✅ Plan for model updates and vocabulary evolution",
    "✅ Monitor for bias in multilingual settings"
]

for item in checklist:
    print(f"  {item}")

print("\n🎉 CONGRATULATIONS!")
print("=" * 40)
print("You now have a comprehensive understanding of modern tokenization techniques!")
print("\n🚀 Next Steps:")
print("  • Experiment with real datasets")
print("  • Integrate with your ML pipeline")
print("  • Explore domain-specific adaptations")
print("  • Stay updated with latest research")
print("\n📚 Keep learning and building amazing NLP applications!")